# Phase 2 — Baseline & metrics under imbalance

Runs the real pipeline and shows why accuracy is the wrong metric.

In [1]:
import sys; sys.path.append('..')
from src.seed import seed_everything; seed_everything(42)


In [2]:
from src.data import get_data
from src.train import engineer, temporal_split, FEATURES

df = engineer(get_data()[0])
train, test = temporal_split(df)
print('fraud rate:', round(train.is_fraud.mean(),4), '| test rows:', len(test))


fraud rate: 0.0055 | test rows: 12000


In [3]:
from sklearn.metrics import accuracy_score, average_precision_score
import numpy as np

# The trivial 'never fraud' model: high accuracy, useless.
y = test.is_fraud.to_numpy()
print('always-negative accuracy:', round(accuracy_score(y, np.zeros_like(y)),4))
print('always-negative PR-AUC :', round(average_precision_score(y, np.zeros_like(y)),4))


always-negative accuracy: 0.9924
always-negative PR-AUC : 0.0076


In [4]:
from src.train import main
results = main()   # trains logistic + gbm, writes reports/evaluation_report.md


{
  "logistic": {
    "pr_auc": 0.11682058349366063,
    "pr_auc_ci": [
      0.0780829225711825,
      0.17880049376957255
    ]
  },
  "gbm": {
    "pr_auc": 0.15222990986383783,
    "pr_auc_ci": [
      0.11058243098700327,
      0.2130724690812259
    ]
  },
  "baseline_always_negative": {
    "pr_auc": 0.007583333333333333
  },
  "operating_point": {
    "model": "gbm",
    "threshold": 0.4214,
    "precision": 0.0356,
    "recall": 0.8681,
    "expected_cost": 16700.0
  },
  "meta": {
    "data_source": "synthetic",
    "n_train": 48000,
    "n_test": 12000,
    "train_fraud_rate": 0.00548,
    "test_fraud_rate": 0.00758
  }
}


**Takeaway:** accuracy > 99% for a model that catches zero fraud. PR-AUC is the honest metric; see `reports/evaluation_report.md` for the full table with bootstrap CIs.